# 🔍 AI-Based Financial Transaction Anomaly Detection

**A step-by-step analysis notebook**

This notebook walks through the complete workflow:
1. Data generation & loading
2. Exploratory Data Analysis (EDA)
3. Preprocessing & Feature Engineering
4. Anomaly Detection (Isolation Forest, One-Class SVM, LOF)
5. Evaluation & comparison
6. Visualisation

## 0. Setup — install dependencies

In [ ]:
# Uncomment if running for the first time
# !pip install pandas numpy scikit-learn matplotlib seaborn plotly streamlit

## 1. Import Libraries

In [ ]:
import os, sys
import warnings
warnings.filterwarnings('ignore')

# Make src/ importable
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:.2f}'.format
print('Libraries loaded ✅')

## 2. Generate / Load Dataset

In [ ]:
# ── Option A: run the generator script ───────────────────────────────────
data_path         = os.path.join('..', 'data', 'transactions.csv')
labeled_data_path = os.path.join('..', 'data', 'transactions_labeled.csv')

if not os.path.exists(data_path):
    print('Generating synthetic data …')
    os.system('python ../data/generate_data.py')

df_raw     = pd.read_csv(data_path)
df_labeled = pd.read_csv(labeled_data_path) if os.path.exists(labeled_data_path) else None

print(f'Raw dataset  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
if df_labeled is not None:
    print(f'Labeled set  : {df_labeled.shape[0]:,} rows  |  anomalies: {df_labeled["is_anomaly"].sum()}')
df_raw.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
df_raw.info()
print('\n=== Summary Statistics ===')
df_raw.describe()

In [ ]:
print('Missing values:')
print(df_raw.isnull().sum())
print('\nValue counts — transaction_type:')
print(df_raw['transaction_type'].value_counts())
print('\nValue counts — merchant_category:')
print(df_raw['merchant_category'].value_counts())

In [ ]:
# Univariate plots
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Transaction amount
axes[0].hist(df_raw['transaction_amount'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount ($)')

# Account balance
axes[1].hist(df_raw['account_balance'], bins=60, color='seagreen', edgecolor='white')
axes[1].set_title('Account Balance Distribution')
axes[1].set_xlabel('Balance ($)')

# Transaction type
tx_counts = df_raw['transaction_type'].value_counts()
axes[2].bar(tx_counts.index, tx_counts.values, color='darkorange', edgecolor='white')
axes[2].set_title('Transaction Types')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Temporal analysis — parse datetime first
df_eda = df_raw.copy()
df_eda['transaction_time'] = pd.to_datetime(df_eda['transaction_time'], errors='coerce')
df_eda['hour']             = df_eda['transaction_time'].dt.hour
df_eda['day_of_week']      = df_eda['transaction_time'].dt.dayofweek
df_eda['month']            = df_eda['transaction_time'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df_eda['hour'].value_counts().sort_index().plot(ax=axes[0], kind='bar', color='steelblue')
axes[0].set_title('Transactions by Hour')

df_eda['day_of_week'].value_counts().sort_index().plot(ax=axes[1], kind='bar', color='seagreen')
axes[1].set_title('Transactions by Day of Week (0=Mon)')

df_eda['month'].value_counts().sort_index().plot(ax=axes[2], kind='bar', color='darkorange')
axes[2].set_title('Transactions by Month')

plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
from data_preprocessing import DataPreprocessor

preprocessor = DataPreprocessor()
df_proc = preprocessor.preprocess(df_raw.copy(), fit=True)

print(f'Preprocessed shape: {df_proc.shape}')
df_proc[['transaction_amount', 'transaction_amount_normalized',
         'account_balance',    'account_balance_normalized',
         'transaction_type',   'transaction_type_encoded',
         'hour', 'is_weekend', 'is_night']].head()

## 5. Feature Engineering

In [ ]:
from feature_engineering import FeatureEngineer

engineer = FeatureEngineer()
df_feat  = engineer.engineer_features(df_proc)

feature_cols      = engineer.get_feature_columns()
available_features = [c for c in feature_cols if c in df_feat.columns]
print(f'Feature columns ({len(available_features)}):', available_features)

df_feat[available_features].describe()

In [ ]:
# Feature correlation heatmap
plt.figure(figsize=(14, 10))
corr = df_feat[available_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Spending deviation distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_feat['spending_deviation'].clip(-5, 5), bins=50, color='steelblue')
axes[0].set_title('Spending Deviation (Z-score, clipped ±5)')
axes[0].set_xlabel('Z-score')

axes[1].hist(df_feat['amount_to_balance_ratio'].clip(0, 5), bins=50, color='darkorange')
axes[1].set_title('Amount-to-Balance Ratio (clipped at 5)')
axes[1].set_xlabel('Ratio')

plt.tight_layout()
plt.show()

## 6. Anomaly Detection

In [ ]:
from anomaly_detection import AnomalyDetector

CONTAMINATION = 0.05   # ~5% anomalies expected

# ── Train Isolation Forest (primary model) ───────────────────────────────
detector_if = AnomalyDetector(model_type='isolation_forest', contamination=CONTAMINATION)
detector_if.fit(df_feat[available_features].fillna(0).values)

df_result_if = detector_if.detect(df_feat, available_features)
print(df_result_if[['transaction_id','transaction_amount','risk_score','is_anomaly_detected']].head(10))

In [ ]:
# ── Attach ground-truth labels if available ──────────────────────────────
if df_labeled is not None and 'transaction_id' in df_result_if.columns:
    label_map = df_labeled.set_index('transaction_id')['is_anomaly'].to_dict()
    df_result_if['is_anomaly'] = df_result_if['transaction_id'].map(label_map).fillna(0).astype(int)
    print('Ground-truth anomalies :', df_result_if['is_anomaly'].sum())
    print('Detected anomalies     :', df_result_if['is_anomaly_detected'].sum())

## 7. Model Comparison — All Three Algorithms

In [ ]:
from anomaly_detection import train_and_evaluate_all_models

models, results = train_and_evaluate_all_models(
    df_feat, available_features, contamination=CONTAMINATION
)

# Add ground-truth if available
if df_labeled is not None:
    for k in results:
        if 'transaction_id' in results[k].columns:
            results[k]['is_anomaly'] = results[k]['transaction_id'].map(label_map).fillna(0).astype(int)
        models[k].evaluate(results[k])

In [ ]:
# Compare anomaly counts across models
comparison = pd.DataFrame({
    'Model': [k.replace('_', ' ').title() for k in results.keys()],
    'Anomalies Detected': [results[k]['is_anomaly_detected'].sum() for k in results],
    'Avg Risk (Anomaly)': [
        results[k][results[k]['is_anomaly_detected']==1]['risk_score'].mean()
        for k in results
    ],
    'Avg Risk (Normal)': [
        results[k][results[k]['is_anomaly_detected']==0]['risk_score'].mean()
        for k in results
    ],
})
comparison

In [ ]:
# Risk score distribution per model
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for ax, (model_name, df_res) in zip(axes, results.items()):
    norm_scores  = df_res[df_res['is_anomaly_detected']==0]['risk_score']
    anom_scores  = df_res[df_res['is_anomaly_detected']==1]['risk_score']
    ax.hist(norm_scores, bins=40, alpha=0.7, color='steelblue', label='Normal')
    ax.hist(anom_scores, bins=20, alpha=0.85, color='crimson',   label='Anomaly')
    ax.set_title(model_name.replace('_', ' ').title())
    ax.set_xlabel('Risk Score')
    ax.legend()

axes[0].set_ylabel('Frequency')
fig.suptitle('Risk Score Distribution by Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Deep Dive — Isolation Forest Results

In [ ]:
from anomaly_detection import visualize_anomalies
visualize_anomalies(df_result_if, output_dir='../results')

In [ ]:
# Suspicious transactions (top 20 by risk score)
suspicious = (
    df_result_if[df_result_if['is_anomaly_detected'] == 1]
    .sort_values('risk_score', ascending=False)
    [['transaction_id','customer_id','transaction_amount',
      'transaction_time','transaction_type','location',
      'account_balance','risk_score']]
    .head(20)
)
suspicious.style.background_gradient(subset=['risk_score'], cmap='RdYlGn_r').format(
    {'transaction_amount':'${:.2f}', 'account_balance':'${:.2f}', 'risk_score':'{:.1f}'}
)

In [ ]:
# Confusion matrix (only if labels exist)
if 'is_anomaly' in df_result_if.columns:
    cm = confusion_matrix(df_result_if['is_anomaly'], df_result_if['is_anomaly_detected'])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred Normal', 'Pred Anomaly'],
                yticklabels=['True Normal', 'True Anomaly'])
    plt.title('Confusion Matrix — Isolation Forest')
    plt.tight_layout()
    plt.show()

    print(classification_report(
        df_result_if['is_anomaly'], df_result_if['is_anomaly_detected'],
        target_names=['Normal', 'Anomaly']
    ))

## 9. Customer Risk Profiling

In [ ]:
# Aggregate risk per customer
customer_risk = (
    df_result_if
    .groupby('customer_id')
    .agg(
        total_transactions   = ('transaction_id', 'count'),
        anomalies_detected   = ('is_anomaly_detected', 'sum'),
        max_risk_score       = ('risk_score', 'max'),
        avg_risk_score       = ('risk_score', 'mean'),
        total_amount         = ('transaction_amount', 'sum'),
        avg_amount           = ('transaction_amount', 'mean'),
    )
    .sort_values('anomalies_detected', ascending=False)
)

print('Top 10 highest-risk customers:')
customer_risk.head(10)

In [ ]:
# Scatter: anomalies vs avg transaction amount per customer
cr = customer_risk.reset_index()
fig = px.scatter(
    cr,
    x='avg_amount', y='max_risk_score',
    size='total_transactions', color='anomalies_detected',
    hover_data=['customer_id', 'total_transactions'],
    color_continuous_scale='RdYlGn_r',
    title='Customer Risk Profile — Avg Spend vs Max Risk Score',
    labels={'avg_amount':'Avg Transaction Amount ($)',
            'max_risk_score':'Max Risk Score',
            'anomalies_detected':'Anomalies'},
)
fig.show()

## 10. Save Outputs

In [ ]:
import os, pickle

os.makedirs('../results', exist_ok=True)
os.makedirs('../models',  exist_ok=True)

# Save suspicious transactions
output_cols = [c for c in ['transaction_id','customer_id','transaction_amount',
                            'transaction_time','transaction_type','merchant_category',
                            'location','account_balance','risk_score']
               if c in df_result_if.columns]

suspicious_all = (
    df_result_if[df_result_if['is_anomaly_detected'] == 1]
    .sort_values('risk_score', ascending=False)[output_cols]
)
out_path = '../results/suspicious_transactions.csv'
suspicious_all.to_csv(out_path, index=False)
print(f'Saved {len(suspicious_all)} suspicious transactions → {out_path}')

# Save trained model
detector_if.save('../models/trained_model.pkl')
preprocessor.save('../models/preprocessor.pkl')

print('\n✅ All outputs saved.')

## 11. Improvement Suggestions

| Area | Suggestion |
|------|------------|
| **More data** | Use a real labelled dataset (e.g., Kaggle Credit Card Fraud) for supervised evaluation |
| **Supervised hybrid** | Once you have labels, add XGBoost / Random Forest on top of the anomaly scores as a second layer |
| **Rolling window features** | Use longer windows (30-day) to capture monthly spending patterns |
| **Graph features** | Model customer–merchant relationships as a graph; isolated graph-edges are suspicious |
| **Autoencoder** | Deep learning reconstruction error gives a powerful unsupervised anomaly signal |
| **Threshold tuning** | Use precision-recall curves to pick the optimal threshold instead of the fixed contamination rate |
| **Real-time scoring** | Wrap `detector.detect()` in a FastAPI endpoint for streaming transaction scoring |
| **Drift detection** | Monitor feature distributions over time; retrain when drift is detected |